# 02b · I gruppi di registrazione
*Blocco 2 · IoT Hub con ADR e DPS. Passo di creazione.*

Un gruppo di registrazione per Hub. Ogni gruppo riferisce la **CA di bootstrap verificata** (per riconoscere il device), la **Issuing CA** di ADR e la **certificate policy** (per emettere il certificato). Con l'allocazione `static` manda tutti i suoi device su un solo Hub.

**Prima:** i notebook 01 e 02a. **Dopo:** il notebook 02c registra i device.

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

Scenario: rg-iot26-dev-uks-001 | namespace ADR adrns-iot-dev-uks | output in D:\Dev\2026\IoTHub\WPC2026\artifacts

## 4 · Un enrollment group per Hub
Il gruppo riferisce la **CA verificata**, la **Issuing CA** e la **certificate policy** di ADR. Con allocazione `static` manda tutti i
suoi device a **un solo Hub**: due device per Hub.

In [ ]:
# Un secondo "create" su un gruppo che esiste gia' risponde 409202 (EnrollmentGroup already exists): i gruppi esistenti si saltano.
$existing = @(az iot dps enrollment-group list --dps-name $dps.Name --resource-group $rg --auth-type login --query '[].enrollmentGroupId' -o tsv --only-show-errors)
foreach ($h in $hubs) {
    $groupId = "eg-$workload-$stage-$($h.Group)"
    if ($existing -contains $groupId) { "$groupId esiste gia': salto la creazione."; continue }
    $groupArgs = @(
        '--dps-name', $dps.Name, '--resource-group', $rg,
        '--enrollment-id', $groupId, '--auth-type', 'key',
        '--root-ca-name', "pcert-$workload-$stage-$($h.Group)", '--allocation-policy', $allocation,
        '--adr-namespace', $adr.Name, '--adr-ca-name', $pki.Issuing, '--adr-cert-policy-name', $pki.Policy
    )
    if ($allocation -eq 'static') { $groupArgs += @('--iot-hubs', "$($h.Name).azure-devices.net") }
    az iot dps enrollment-group create @groupArgs -o none
    Assert-Az "Creazione dell'enrollment group $groupId"
}

eg-iot26-dev-robots esiste gia': salto la creazione.

eg-iot26-dev-trucks esiste gia': salto la creazione.

In [ ]:
az iot dps enrollment-group list --dps-name $dps.Name --resource-group $rg --auth-type login --query '[].{gruppo:enrollmentGroupId, attestation:attestation.type, allocazione:allocationPolicy}' -o table
az iot dps certificate list --dps-name $dps.Name --resource-group $rg --query 'value[].{ca:name, verificata:properties.isVerified}' -o table

Gruppo               Attestation    Allocazione

-------------------  -------------  -------------

eg-iot26-dev-robots  x509           static

eg-iot26-dev-trucks  x509           static

Ca                      Verificata

----------------------  ------------

pcert-iot26-dev-robots  True

pcert-iot26-dev-trucks  True

Prossimo passo: **02c**, i device.